In [1]:
from utils import *

In [ ]:
def get_homogenized_elastic_modulus(**kwargs):
    """
    kwargs: {'section': string, flange/web/matrix: matrix, 'dim': []}
    - for T stringer's dimension: [0: flange width, 1: flange thickness, 2: web thickness, 3: web height]
    - for Omega stringer's dimension: [0: upper flange width, 1: web height, 2: lower flange width, 3: thickness]
    engineering constant for axial loading is used to get homogenized E
    """
    # TODO: check if mid-line value is used for this computation
    if kwargs['section'] == 'T':
        # two different ABD matrices for flange and web
        ABD_flange = kwargs['ABD_flange']
        ABD_web = kwargs['ABD_web']
        ABD_inv_web = np.linalg.inv(ABD_web)

        # compute the mid-line value
        flange_bm = kwargs['dim'][0]
        flange_t = kwargs['dim'][1]
        web_t = kwargs['dim'][2] / 2
        web_bm = kwargs['dim'][3] - web_t / 2

        E_flange_x = ABD_flange[0][0] / flange_t # flange: constrained
        E_web_x = 1 / (ABD_inv_web[0][0] * web_t) # web: free

        # compute mid-line area
        flange_area = flange_bm * flange_t
        web_area = web_bm * web_t

        # compute the average elastic modules
        E_avg = (E_flange_x * flange_area + E_web_x * web_area) / (flange_area + web_area)

    else: # kwargs['section'] == 'Omega'
        # one uniform laminate for the omega section
        ABD_mat = kwargs['ABD_omega']
        ABD_inv_mat = np.linalg.inv(ABD_mat)

        # compute the mid-line value
        t_omega = kwargs['dim'][3]
        up_flange_bm = kwargs['dim'][0] + t_omega / 2
        web_hm = kwargs['dim'][1] - t_omega
        low_flange_bm = kwargs['dim'][2] - t_omega

        E_up_flange_x = ABD_mat[0][0] / t_omega # upper flange: constrained
        E_web_x = 1 / (ABD_inv_mat[0][0] * t_omega) # web: free
        E_low_flange_x = 1 / (ABD_inv_mat[0][0] * t_omega) # lower flange: free

        # compute mid-line area
        up_flange_area = up_flange_bm * t_omega
        web_area = web_hm * t_omega
        low_flange_area = low_flange_bm * t_omega

        # compute the average elastic modules
        sum_area = 2 * up_flange_area + 2 * web_area + low_flange_area
        E_avg = (2 * E_up_flange_x * up_flange_area + 2 * E_web_x * web_area + E_low_flange_x * low_flange_area) / sum_area

    return E_avg

In [2]:
Q_mat = get_Q_matrix()
Q_mat

matrix([[127778.83499817,   3121.35936927,      0.        ],
        [  3121.35936927,   9754.24802897,      0.        ],
        [     0.        ,      0.        ,   6213.6       ]])

Trying to compute the homogenized elastic modules of a T-stringer

In [ ]:
# TODO: DO CHECK THE SEQUENCE
stringer_seq = ['T', 'T', 'O', 'T', 'O', 'T', 'T', 'O', 'O', 'O']

flange_stack_s = [45, 45, -45, -45, 0, 0, 90, 90]
web_stack_s = [-45, -45, 45, 45, 0, 0, 90, 90]
omega_stack_s = []
t = 0.25

# compute the for T
dim_t = []
ABD_flange = get_ABD_mat(Q_mat, flange_stack_s, t)
ABD_web = get_ABD_mat(Q_mat, web_stack_s, t)
E_avg = get_homogenized_elastic_modulus(ABD_flange=ABD_flange, ABD_web=ABD_web, dim=dim_t)

# compute the for
dim_omega = []
ABD_omega = get_ABD_mat(Q_mat, omega_stack_s, t)
E_avg = get_homogenized_elastic_modulus(ABD_omega=ABD_omega, dim=dim_omega)